In [1]:
# 下载 BEiT-Large-224 预训练权重 (约 1.1 GB)
!wget -q --show-progress --no-check-certificate https://hf-mirror.com/timm/beit_large_patch16_224.in22k_ft_in22k_in1k/resolve/main/model.safetensors -O beit_large.safetensors

import os
size_mb = os.path.getsize('beit_large.safetensors') / (1024 * 1024)
print(f"📦 检查文件大小: {size_mb:.2f} MB")
if size_mb > 1000:
    print("✅ BEiT-Large 预训练权重下载真正完成！")
else:
    print("🚨 警告：文件大小不对，可能被拦截了，请重新运行或检查网络！")

beit_large.safetens 100%[===================>]   1.14G  4.20MB/s    in 4m 45s  
📦 检查文件大小: 1168.45 MB
✅ BEiT-Large 预训练权重下载真正完成！


In [1]:
import os
import gc
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import log_loss, confusion_matrix
import timm
from transformers import get_cosine_schedule_with_warmup

# ==========================================
# ⚙️ 1. 全局配置参数 (BEiT-Large 伪标签微调版)
# ==========================================
CSV_PATH = 'dataset/driver_imgs_list.csv'

MODEL_NAME = 'beit_large_patch16_224.in22k_ft_in22k_in1k'
IMG_SIZE = 224      # BEiT 规定尺寸
EPOCHS = 3          # ✅ 伪标签微调只需要 3 轮
BATCH_SIZE = 32     # BEiT-Large 极度吃显存，保持 32
ACCUMULATION_STEPS = 1  
NUM_SPLITS = 5
TARGET_FOLDS = [0, 1, 2, 3, 4]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ A100/5090 算力榨干配置
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

gc.collect()
torch.cuda.empty_cache()

# ==========================================
# 📊 2. 数据划分 (完全复刻 Eval_Pipeline 贪心隔离策略)
# ==========================================
def generate_balanced_folds(csv_path, n_splits=5):
    df = pd.read_csv(csv_path).reset_index(drop=True)
    if 'label_int' not in df.columns:
        df['label_int'] = df['classname'].str.extract(r'(\d+)').astype(int)

    driver_counts = df.groupby('subject').size().sort_values(ascending=False)
    fold_totals = np.zeros(n_splits)
    fold_groups = [[] for _ in range(n_splits)]

    for subject, count in driver_counts.items():
        min_fold_idx = np.argmin(fold_totals)
        fold_groups[min_fold_idx].append(subject)
        fold_totals[min_fold_idx] += count

    df['fold'] = -1
    for i, subjects in enumerate(fold_groups):
        df.loc[df['subject'].isin(subjects), 'fold'] = i

    print("\n" + "="*50)
    print("            📊 5 折原始数据分布情况")
    print("="*50)
    for i in range(n_splits):
        print(f"Fold {i}  |  驾驶员数量: {len(fold_groups[i]):2d}  |  图片总数: {int(fold_totals[i]):4d}")
    print("="*50 + "\n")
    return df

# ==========================================
# 🖼️ 3. 数据集与增强
# ==========================================
def get_train_transforms(img_size=224):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, p=0.6),
        A.Affine(translate_percent=(-0.05, 0.05), scale=(0.95, 1.05), rotate=(-10, 10), p=0.5),
        A.GaussNoise(p=0.4),
        A.CoarseDropout(num_holes_range=(1, 8), hole_height_range=(1, 16), hole_width_range=(1, 16), fill=0, p=0.4),
        A.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]), 
        ToTensorV2()
    ])

def get_valid_transforms(img_size=224):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ToTensorV2()
    ])

class DriverDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # 👑 智能路由：判断图片来源于原始训练集还是伪标签测试集
        if row['subject'] == 'pseudo_test':
            img_path = os.path.join('./dataset/imgs/test_cropped_v2', row['img'])
        else:
            img_path = os.path.join('./dataset/imgs/train_cropped_v2', row['classname'], row['img'])
        
        image = cv2.imread(img_path)
        
        # 防御性回退机制
        if image is None:
            if row['subject'] == 'pseudo_test':
                image = cv2.imread(os.path.join('./dataset/imgs/test', row['img']))
            else:
                image = cv2.imread(os.path.join('./dataset/imgs/train', row['classname'], row['img']))
        
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform:
            image = self.transform(image=image)['image']
        return image, row['label_int']

# ==========================================
# 🛠️ 4. 评估工具
# ==========================================
def plot_confusion_matrix(y_true, y_pred, fold_idx, save_dir):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues')
    plt.title(f'Fold {fold_idx} Confusion Matrix (Pseudo)')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(os.path.join(save_dir, f'pseudo_cm_fold_{fold_idx}.png'), dpi=300, bbox_inches='tight')
    plt.close()

# ==========================================
# 🚀 5. 主干微调流程
# ==========================================
def main():
    base_dir = 'models/beit'
    os.makedirs(base_dir, exist_ok=True)

    full_df = generate_balanced_folds(CSV_PATH, NUM_SPLITS)
    oof_preds = np.zeros((len(full_df), 10))
    
    # 👑 加载提纯后的伪标签数据
    pseudo_df = pd.read_csv('pseudo_labels.csv')

    for fold in TARGET_FOLDS:
        print(f"\n{'='*40}\n🌟 开始微调 BEiT Fold {fold} (带伪标签加持) 🌟\n{'='*40}")

        # 验证集保持绝对纯净，不能有伪标签穿越！
        val_df = full_df[full_df['fold'] == fold]
        
        # 训练集混合伪标签
        original_train = full_df[full_df['fold'] != fold]
        train_df = pd.concat([original_train, pseudo_df], axis=0).reset_index(drop=True)
        
        print(f"  -> 当前折训练集数量: {len(train_df)} (包含 {len(pseudo_df)} 张伪标签)")
        print(f"  -> 当前折验证集数量: {len(val_df)}")

        # ✅ 修复：删除了 TRAIN_DIR 参数，启用智能路由
        train_loader = DataLoader(
            DriverDataset(train_df, transform=get_train_transforms(IMG_SIZE)),
            batch_size=BATCH_SIZE, shuffle=True, num_workers=8, pin_memory=True, drop_last=True
        )
        val_loader = DataLoader(
            DriverDataset(val_df, transform=get_valid_transforms(IMG_SIZE)),
            batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True
        )

        # 1. 建立空模型
        model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=10, drop_path_rate=0.1)
        
        # 2. 👑 加载上一轮最优秀的模型作为起点 (而不是最原始的 safetensors)
        previous_best_weight = os.path.join(base_dir, f"best_model_beit_fold_{fold}.pth")
        print(f"📥 正在继承上一轮优秀的 BEiT 权重: {previous_best_weight}")
        model.load_state_dict(torch.load(previous_best_weight, map_location=DEVICE, weights_only=True))
        model.to(DEVICE)

        # 3. 极小学习率微调 (取消了原本两极分化的学习率，因为模型已经收敛，这里追求全局平滑微调)
        optimizer = AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)

        class_weights = torch.tensor([1.0]*9 + [1.5], dtype=torch.float).to(DEVICE)
        criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
        scaler = torch.amp.GradScaler('cuda')

        total_steps = (len(train_loader) // ACCUMULATION_STEPS) * EPOCHS
        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(total_steps * 0.1),
            num_training_steps=total_steps
        )

        best_val_loss = float('inf')
        # ✅ 保存名字加上 pseudo_ 前缀，保护原模型不被覆盖
        save_path = os.path.join(base_dir, f"pseudo_best_model_beit_fold_{fold}.pth")
        EARLY_STOP_PATIENCE = 2
        epochs_no_improve = 0

        for epoch in range(EPOCHS):
            model.train()
            train_loss = 0.0

            pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")
            for i, (images, labels) in enumerate(pbar):
                images, labels = images.to(DEVICE), labels.to(DEVICE)

                with torch.amp.autocast('cuda'):
                    outputs = model(images)
                    loss = criterion(outputs, labels) / ACCUMULATION_STEPS

                scaler.scale(loss).backward()
                train_loss += loss.item() * ACCUMULATION_STEPS

                if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(train_loader):
                    # 梯度裁剪依然保留，防止伪标签偶尔带来的毛刺梯度冲毁模型
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad()
                    scheduler.step()

                pbar.set_postfix({'loss': f"{loss.item()*ACCUMULATION_STEPS:.4f}"})

            model.eval()
            val_loss = 0.0
            fold_preds_list, fold_labels = [], []

            vbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Valid]")
            with torch.no_grad():
                for images, labels in vbar:
                    images, labels = images.to(DEVICE), labels.to(DEVICE)
                    with torch.amp.autocast('cuda'):
                        outputs = model(images)
                        loss = criterion(outputs, labels)

                    val_loss += loss.item()
                    fold_preds_list.append(outputs.softmax(dim=1).cpu().numpy())
                    fold_labels.extend(labels.cpu().numpy())
                    vbar.set_postfix({'v_loss': f"{loss.item():.4f}"})

            avg_val_loss = val_loss / len(val_loader)
            current_fold_preds = np.concatenate(fold_preds_list, axis=0)

            if avg_val_loss < best_val_loss:
                print(f"✅ 验证集 Loss 降低: {best_val_loss:.4f} -> {avg_val_loss:.4f}，保存微调模型。")
                best_val_loss = avg_val_loss
                torch.save(model.state_dict(), save_path)
                epochs_no_improve = 0 
                
                oof_preds[val_df.index] = current_fold_preds
                best_labels = fold_labels
                best_preds = np.argmax(current_fold_preds, axis=1)
            else:
                epochs_no_improve += 1
                print(f"⚠️ 验证集 Loss 未降低 ({epochs_no_improve}/{EARLY_STOP_PATIENCE})")
                if epochs_no_improve >= EARLY_STOP_PATIENCE:
                    print(f"🛑 连续 {EARLY_STOP_PATIENCE} 轮未下降，触发早停！")
                    break

        print("📸 正在生成微调后的混淆矩阵...")
        plot_confusion_matrix(best_labels, best_preds, fold, base_dir)

        del model, optimizer, train_loader, val_loader
        gc.collect()
        torch.cuda.empty_cache()

    print("\n🎉 所有 Fold 微调完毕！保存 OOF 预测结果...")
    np.save(os.path.join(base_dir, "pseudo_oof_preds_beit.npy"), oof_preds)

    final_labels = full_df['label_int'].values
    final_log_loss = log_loss(final_labels, oof_preds)
    print(f"🔥 伪标签加持后 Final OOF Log Loss (BEiT-Large): {final_log_loss:.4f}")

if __name__ == "__main__":
    main()

/root/miniconda3/lib/python3.12/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info The read operation timed out
  data = fetch_version_info()



            📊 5 折原始数据分布情况
Fold 0  |  驾驶员数量:  5  |  图片总数: 4407
Fold 1  |  驾驶员数量:  5  |  图片总数: 4475
Fold 2  |  驾驶员数量:  5  |  图片总数: 4364
Fold 3  |  驾驶员数量:  6  |  图片总数: 4704
Fold 4  |  驾驶员数量:  5  |  图片总数: 4474


🌟 开始微调 BEiT Fold 0 (带伪标签加持) 🌟
  -> 当前折训练集数量: 75645 (包含 57628 张伪标签)
  -> 当前折验证集数量: 4407
📥 正在继承上一轮优秀的 BEiT 权重: models/beit/best_model_beit_fold_0.pth


Epoch 1/3 [Train]:   0%|          | 1/2363 [00:02<1:19:54,  2.03s/it, loss=0.7168]


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 31.36 GiB of which 18.38 MiB is free. Process 1553 has 21.87 GiB memory in use. Including non-PyTorch memory, this process has 9.45 GiB memory in use. Of the allocated memory 8.43 GiB is allocated by PyTorch, and 400.85 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [2]:
import os
import torch
import pandas as pd
import numpy as np
import timm
from tqdm.notebook import tqdm
import cv2
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ==========================================
# ⚙️ 1. 基础配置 (BEiT-Large 巨兽版)
# ==========================================
# 🔥 5090 专属加速魔法：开启 TF32 核心
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

FOLDS = [0, 1, 2, 3, 4]
SAMPLE_SUBMISSION_PATH = './dataset/sample_submission.csv'
TEST_DIR = 'dataset/imgs/test_cropped_v2'

# ✅ BEiT 专属路径与模型名称
SAVE_DIR = 'models/beit'
MODEL_NAME = 'beit_large_patch16_224.in22k_ft_in22k_in1k'
WEIGHT_PATH_TEMPLATE = os.path.join(SAVE_DIR, 'pseudo_best_model_beit_fold_{}.pth')
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ⚠️ 极其关键：尺寸必须是 224
IMG_SIZE = 224      
# BEiT-Large 参数量高达 307M，纯推理时 5090 显存很大，但为了防爆，设为 64 或 128
BATCH_SIZE = 128    
NUM_WORKERS = 8

# ==========================================
# 🖼️ 2. 构建测试集 DataLoader (按 CSV 严格保序)
# ==========================================
class TestDriverDataset(Dataset):
    def __init__(self, csv_path, test_dir):
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"🚨 找不到官方提交模板: {csv_path}")
        self.df = pd.read_csv(csv_path)
        self.test_dir = test_dir
        
        self.image_names = self.df['img'].values

        # ⚠️ 极其关键：BEiT 专属的归一化均值和方差，绝对不能错！
        self.transform = A.Compose([
            A.Resize(IMG_SIZE, IMG_SIZE),
            A.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
            ToTensorV2()
        ])

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names[idx]
        img_path = os.path.join(self.test_dir, img_name)
        
        image = cv2.imread(img_path)
        if image is None:
            fallback_path = os.path.join('./dataset/imgs/test', img_name)
            image = cv2.imread(fallback_path)
            if image is None:
                raise FileNotFoundError(f"找不到图片: {img_path}")
                
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image_tensor = self.transform(image=image)['image']

        return image_tensor, img_name

# ==========================================
# 🚀 3. 核心预测与 5折融合 (Ensemble)
# ==========================================
def generate_submission():
    print(f"🚀 启动 BEiT-Large 预测流程！使用设备: {DEVICE}")

    test_dataset = TestDriverDataset(SAMPLE_SUBMISSION_PATH, TEST_DIR)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    print(f"📦 共抓取 {len(test_dataset)} 张测试图片。")

    all_fold_preds = []

    for fold in FOLDS:
        weight_path = WEIGHT_PATH_TEMPLATE.format(fold)
        print(f"\n👨‍⚖️ 正在准备第 {fold} 号 BEiT 模型...")

        if not os.path.exists(weight_path):
            print(f"⚠️ 找不到权重文件 {weight_path}，跳过此折！")
            continue

        # 1. 实例化干净的空模型
        model = timm.create_model(MODEL_NAME, pretrained=False, num_classes=10)
        model.to(DEVICE)
        
        # 2. 加载训练好的权重
        state_dict = torch.load(weight_path, map_location=DEVICE, weights_only=True)
        model.load_state_dict(state_dict, strict=True)
        model.eval()

        # 3. 编译加速 (BEiT 这种大 Transformer 极其吃这个加速)
        if int(torch.__version__.split('.')[0]) >= 2:
            print("⚡ 启用 torch.compile 加速大模型推理...")
            model = torch.compile(model)

        fold_preds = []
        with torch.no_grad():
             # 使用 AMP 半精度推理，省显存提速
             with torch.amp.autocast('cuda'):
                pbar = tqdm(test_loader, desc=f"⚡ Fold {fold} 推理中")
                for images, _ in pbar:
                    images = images.to(DEVICE)
                    outputs = model(images)
                    probs = torch.softmax(outputs, dim=1)
                    fold_preds.append(probs.cpu().numpy())

        fold_preds = np.concatenate(fold_preds, axis=0)
        all_fold_preds.append(fold_preds)
        
        del model
        torch.cuda.empty_cache()

    if not all_fold_preds:
        print("❌ 没有找到任何有效的权重文件，推理终止。")
        return

    # ==========================================
    # 🤝 4. 终极平均融合生成提交文件
    # ==========================================
    print(f"\n🤝 正在对找到的 {len(all_fold_preds)} 折 BEiT 模型进行融合...")
    all_fold_preds = np.array(all_fold_preds) 
    final_preds = np.mean(all_fold_preds, axis=0) 

    print("📝 正在生成 BEiT 的 CSV 提交文件...")
    img_filenames = test_dataset.image_names

    df_submit = pd.DataFrame(final_preds, columns=[f'c{i}' for i in range(10)])
    df_submit.insert(0, 'img', img_filenames) 

    submit_filename = os.path.join(SAVE_DIR, 'submission_beit_5fold.csv')
    df_submit.to_csv(submit_filename, index=False)
    print(f"🎉 大功告成！BEiT 预测文件已保存为: {submit_filename}")

if __name__ == '__main__':
    generate_submission()

🚀 启动 BEiT-Large 预测流程！使用设备: cuda
📦 共抓取 79726 张测试图片。

👨‍⚖️ 正在准备第 0 号 BEiT 模型...
⚡ 启用 torch.compile 加速大模型推理...


⚡ Fold 0 推理中:   0%|          | 0/623 [00:00<?, ?it/s]


👨‍⚖️ 正在准备第 1 号 BEiT 模型...
⚡ 启用 torch.compile 加速大模型推理...


⚡ Fold 1 推理中:   0%|          | 0/623 [00:00<?, ?it/s]


👨‍⚖️ 正在准备第 2 号 BEiT 模型...
⚡ 启用 torch.compile 加速大模型推理...


...


👨‍⚖️ 正在准备第 3 号 BEiT 模型...
⚡ 启用 torch.compile 加速大模型推理...


⚡ Fold 3 推理中:   0%|          | 0/623 [00:00<?, ?it/s]


👨‍⚖️ 正在准备第 4 号 BEiT 模型...
⚡ 启用 torch.compile 加速大模型推理...


⚡ Fold 4 推理中:   0%|          | 0/623 [00:00<?, ?it/s]


🤝 正在对找到的 5 折 BEiT 模型进行融合...
📝 正在生成 BEiT 的 CSV 提交文件...
🎉 大功告成！BEiT 预测文件已保存为: models/beit/submission_beit_5fold.csv


In [2]:
import os
import torch
import numpy as np
import pandas as pd
import timm
from tqdm.notebook import tqdm
import cv2
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TEST_DIR = 'dataset/imgs/test_cropped_v2'
SAMPLE_SUBMISSION_PATH = './dataset/sample_submission.csv'
IMG_SIZE = 224  # BEiT 尺寸
BATCH_SIZE = 128
FOLDS = [0, 1, 2, 3, 4]  # 👈 设定你要提取的折数

# 加载测试集路径
df_test = pd.read_csv(SAMPLE_SUBMISSION_PATH)
image_names = df_test['img'].values

# BEiT 专属归一化
transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ToTensorV2()
])

class TestFeatureDataset(Dataset):
    def __len__(self): return len(image_names)
    def __getitem__(self, idx):
        img_path = os.path.join(TEST_DIR, image_names[idx])
        image = cv2.imread(img_path)
        if image is None:
            image = cv2.imread(os.path.join('./dataset/imgs/test', image_names[idx]))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        return transform(image=image)['image']

print("🚀 准备提取 BEiT 5折骨干特征...")
test_loader = DataLoader(TestFeatureDataset(), batch_size=BATCH_SIZE, shuffle=False, num_workers=8)

# 用于存放 5 折特征的列表
all_fold_features = []

for fold in FOLDS:
    print(f"\n👨‍⚖️ 正在加载 BEiT Fold {fold} 模型...")
    
    # 1. 每次循环创建一个干净的模型
    model = timm.create_model('beit_large_patch16_224.in22k_ft_in22k_in1k', pretrained=False, num_classes=10)
    
    # 2. 读取对应折的权重
    weight_path = f'models/beit/pseudo_best_model_beit_fold_{fold}.pth'
    if not os.path.exists(weight_path):
        print(f"⚠️ 找不到权重 {weight_path}，跳过此折！")
        continue
        
    model.load_state_dict(torch.load(weight_path, map_location=DEVICE, weights_only=True))

    # 3. 切除分类头！
    model.reset_classifier(0)
    model.to(DEVICE)
    model.eval()

    # 编译加速 (如果支持)
    if int(torch.__version__.split('.')[0]) >= 2:
        model = torch.compile(model)

    fold_features = []
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            for images in tqdm(test_loader, desc=f"💎 提取 Fold {fold} 特征"):
                features = model(images.to(DEVICE))
                fold_features.append(features.cpu().numpy())

    # 拼接当前折的所有批次特征 -> 形状 (79726, 1024)
    current_fold_final_features = np.concatenate(fold_features, axis=0)
    all_fold_features.append(current_fold_final_features)
    
    # 清理显存，防止 5 折跑下来爆显存
    del model
    torch.cuda.empty_cache()

if not all_fold_features:
    raise ValueError("🚨 没有成功提取到任何特征！")

# 🎯 魔法：对 5 折的特征矩阵求算术平均！
print("\n🤝 正在对 5 折特征进行算术平均融合...")
all_fold_features = np.array(all_fold_features) # 形状: (5, 79726, 1024)
final_avg_features = np.mean(all_fold_features, axis=0) # 形状: (79726, 1024)

# 覆盖保存为单一的 npy 文件
save_path = 'models/test_features_beit.npy'
np.save(save_path, final_avg_features)
print(f"✅ BEiT 5折特征融合完毕！矩阵形状: {final_avg_features.shape}")
print(f"📁 已安全保存至: {save_path}")

🚀 准备提取 BEiT 5折骨干特征...

👨‍⚖️ 正在加载 BEiT Fold 0 模型...


💎 提取 Fold 0 特征:   0%|          | 0/623 [00:00<?, ?it/s]


👨‍⚖️ 正在加载 BEiT Fold 1 模型...


💎 提取 Fold 1 特征:   0%|          | 0/623 [00:00<?, ?it/s]


👨‍⚖️ 正在加载 BEiT Fold 2 模型...


...


👨‍⚖️ 正在加载 BEiT Fold 3 模型...


💎 提取 Fold 3 特征:   0%|          | 0/623 [00:00<?, ?it/s]


👨‍⚖️ 正在加载 BEiT Fold 4 模型...


💎 提取 Fold 4 特征:   0%|          | 0/623 [00:00<?, ?it/s]


🤝 正在对 5 折特征进行算术平均融合...
✅ BEiT 5折特征融合完毕！矩阵形状: (79726, 1024)
📁 已安全保存至: models/test_features_beit.npy
